# Milk case study: organic *vs* conventional cow milk

This notebook recomputes everything in the article that concerns the milk cohort, starting from the
deposited feature table (`data/milk_10mz.pkl`, 10 m/z × 2 min grid). We go through the classification of
Tables 1 and 5, the class-wise metrics, the batch and sample-size analyses of the Supporting Information,
the bins discussed in the text (bin 183 first, then the other regions of Table 2; Figure 2 and
Figure S3), the downstream inspection of the most important bins and the coordinate check.
Section 8 goes back to the raw files when they are available (`MINUT_RAW/milk/**/*.mzXML`, 166 of the 177
samples); when they are not, it reuses the per-sample table written by the last raw pass.

We ran it with the pinned environment of `requirements.txt` (Python 3.11, scikit-learn 1.3.0, NumPy 1.26.4).
Random forests differ between scikit-learn versions, so the exact numbers need these versions.

In [1]:
import json, os, platform, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, sklearn
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import fisher_exact
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, cross_val_score, permutation_test_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

sys.path.insert(0, str(Path.cwd()))
import minut as mn
from minut import GRIDS

warnings.filterwarnings("ignore")
GRID = GRIDS["milk"]; NB = int(np.prod(mn.grid_shape(GRID)))          # 630 bins, 1260 descriptors (m/z, RT)
OUT = mn.RESULTS / "milk"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print(f"Python {platform.python_version()}, numpy {np.__version__}, scipy {scipy.__version__}, scikit-learn {sklearn.__version__}")
print("raw milk files:", mn.raw_available("milk"))

Python 3.11.9, numpy 1.26.4, scipy 1.16.3, scikit-learn 1.3.0
raw milk files: True


## 1. The deposited table and the cohort

The table holds 199 records for 193 sample identifiers, acquired in four batches. As in the article, we
keep the first occurrence of each identifier and the two large batches (20220215 and 20220216), which
leaves 177 samples, 48 organic and 129 conventional, split 141/36 with
`train_test_split(test_size=0.2, random_state=42)`.

In [2]:
raw = mn.milk_table()
first = raw.drop_duplicates("echantillon")
comp = pd.crosstab(raw.batch, raw.categ).rename(columns={"b": "organic", "c": "conventional"}).join(
    pd.crosstab(first.batch, first.categ).rename(columns={"b": "organic", "c": "conventional"}), lsuffix="_records", rsuffix="_first_occurrence")
dup = raw[raw.duplicated("echantillon", keep=False)].sort_values(["echantillon", "batch"]).groupby("echantillon").agg(
    batches=("batch", ";".join), labels=("categ", ";".join))
dup["label_conflict"] = dup.labels.str.split(";").apply(lambda l: len(set(l)) > 1)
print(f"{len(raw)} records, {raw.echantillon.nunique()} identifiers; {len(dup)} identifiers in two batches, {int(dup.label_conflict.sum())} with conflicting labels")
display(comp)
comp.to_csv(OUT / "milk_batch_composition.csv"); dup.to_csv(OUT / "milk_duplicated_identifiers.csv")

X, y, train, test = mn.load_milk()
print(f"cohort: {len(y)} samples ({int(y.sum())} organic, {int((y == 0).sum())} conventional); training {len(train)}, test {len(test)} ({int((y[test] == 0).sum())} conventional, {int(y[test].sum())} organic)")

199 records, 193 identifiers; 6 identifiers in two batches, 1 with conflicting labels

categ,organic_records,conventional_records,organic_first_occurrence,conventional_first_occurrence
batch,,,,
20220210,2,2,2,2
20220214,6,6,6,6
20220215,13,47,12,47
20220216,38,85,36,82


cohort: 177 samples (48 organic, 129 conventional); training 141, test 36 (27 conventional, 9 organic)


## 2. Classification (Table 1, MINUT row; Table 5)

Ten-fold cross-validation on the training set, with the 100 most important descriptors re-selected inside
each fold (selector forest seed 0, classifier seed 2024). The final model keeps the descriptors whose
importance reaches the 101st largest value; the threshold of the original notebook was inclusive, so 101
descriptors come out, and we kept that behaviour. The model is scored on the 36 held-out samples:
accuracy, ROC AUC, a percentile bootstrap of the test set (B = 1000) and the permutation test of the
original analysis (100 label permutations of the test set, five folds, on the selected descriptors).

In [3]:
cv = cross_val_score(Pipeline([("select", mn.GiniSelector(100, 0)), ("forest", RandomForestClassifier(n_estimators=100, random_state=2024))]),
                     X[train], y[train], cv=10)
keep = mn.threshold_selection(X[train], y[train], 100, random_state=0)
model, pred, proba, sc = mn.holdout(X, y, train, test, keep, seed=2024)
_, _, perm_p = permutation_test_score(model, X[test][:, keep], y[test], cv=5, n_permutations=100, scoring="accuracy", random_state=42)
summary_ = mn.classification_summary(y[test], pred, proba); classification = summary_
boot_bal = [balanced_accuracy_score(*resample(y[test], pred, random_state=i)) for i in range(1000)]
table1 = dict(cv_accuracy=cv.mean(), cv_sd=cv.std(), permutation_p=perm_p, **sc)
print(f"training 10-fold CV accuracy {cv.mean():.4f} ± {cv.std():.4f}")
print(f"test: accuracy {sc['test_accuracy']:.4f} ({classification['tn'] + classification['tp']}/{len(test)}), AUC {sc['test_auc']:.4f}, bootstrap {sc['bootstrap_accuracy']:.4f} ± {sc['bootstrap_sd']:.4f}, permutation p {perm_p:.4f}, {sc['n_features']} descriptors")
print(f"confusion [[tn fp] [fn tp]] = [[{classification['tn']} {classification['fp']}] [{classification['fn']} {classification['tp']}]]; balanced accuracy {classification['balanced_accuracy']:.3f}; "
      f"sensitivity conventional {classification['sensitivity_conventional']:.2f} (Wilson {classification['sensitivity_conventional_wilson95'][0]:.2f}-{classification['sensitivity_conventional_wilson95'][1]:.2f}), "
      f"organic {classification['sensitivity_organic']:.2f} ({classification['sensitivity_organic_wilson95'][0]:.2f}-{classification['sensitivity_organic_wilson95'][1]:.2f}); majority baseline {classification['majority_baseline']:.2f}")
pd.DataFrame([table1]).to_csv(OUT / "milk.csv", index=False, float_format="%.6f")
classification.update(bootstrap_balanced_accuracy=float(np.mean(boot_bal)), bootstrap_balanced_sd=float(np.std(boot_bal)), n_selected_descriptors=int(keep.sum()))
pd.DataFrame([classification]).to_csv(OUT / "milk_test_metrics.csv", index=False)
pd.DataFrame(dict(y_true=y[test], y_pred=pred, proba_organic=proba)).to_csv(OUT / "milk_test_predictions.csv", index=False)


training 10-fold CV accuracy 0.9219 ± 0.0386
test: accuracy 0.9444 (34/36), AUC 0.9733, bootstrap 0.9454 ± 0.0378, permutation p 0.0099, 101 descriptors
confusion [[tn fp] [fn tp]] = [[27 0] [2 7]]; balanced accuracy 0.889; sensitivity conventional 1.00 (Wilson 0.88-1.00), organic 0.78 (0.45-0.94); majority baseline 0.75


Out-of-fold predictions over the whole cohort: ten stratified folds, the selection refitted in each
training fold, and five different fold assignments (seeds 0 to 4).

In [4]:
rows = []
for seed in range(5):
    oof = np.zeros(len(y), int)
    for tr, te in StratifiedKFold(10, shuffle=True, random_state=seed).split(X, y):
        k_ = mn.threshold_selection(X[tr], y[tr], 100, random_state=0)
        oof[te] = RandomForestClassifier(n_estimators=100, random_state=2024).fit(X[tr][:, k_], y[tr]).predict(X[te][:, k_])
    rows.append(dict(seed=seed, accuracy=accuracy_score(y, oof), balanced_accuracy=balanced_accuracy_score(y, oof),
                     sens_conventional=((oof == 0) & (y == 0)).sum() / (y == 0).sum(), sens_organic=((oof == 1) & (y == 1)).sum() / (y == 1).sum()))
oof_table = pd.DataFrame(rows); display(oof_table.round(3)); oof_table.to_csv(OUT / "milk_oof_177.csv", index=False, float_format="%.6f")

,seed,accuracy,balanced_accuracy,sens_conventional,sens_organic
0,0,0.932,0.908,0.961,0.854
1,1,0.932,0.901,0.969,0.833
2,2,0.932,0.901,0.969,0.833
3,3,0.938,0.905,0.977,0.833
4,4,0.938,0.912,0.969,0.854


## 3. Sensitivity to the batch selection (Supporting Information, Table S7)

The same protocol (80/20 split with seed 42, in-fold selection of 100 descriptors, ten-fold CV, threshold
selection and forest seed 2024) applied to four versions of the cohort, to see how much the choice of the
two large batches matters.

In [5]:
def protocol(df, label):
    Xc, yc = mn.milk_matrix(df)
    tr, te = train_test_split(np.arange(len(yc)), test_size=0.2, random_state=42)
    cvc = cross_val_score(Pipeline([("select", mn.GiniSelector(100, 0)), ("forest", RandomForestClassifier(n_estimators=100, random_state=2024))]), Xc[tr], yc[tr], cv=10)
    kc = mn.threshold_selection(Xc[tr], yc[tr], 100, random_state=0)
    _, pr, pb, s = mn.holdout(Xc, yc, tr, te, kc, seed=2024, n_boot=0)
    c = mn.classification_summary(yc[te], pr, pb)
    return dict(cohort=label, n=len(yc), n_train=len(tr), n_test=len(te), cv_accuracy=cvc.mean(), cv_sd=cvc.std(), test_accuracy=s["test_accuracy"],
                test_balanced_accuracy=c["balanced_accuracy"], test_auc=s["test_auc"], test_majority_baseline=c["majority_baseline"], test_n_conventional=c["n_negative"],
                test_n_organic=c["n_positive"], sens_conventional=c["sensitivity_conventional"], sens_organic=c["sensitivity_organic"],
                confusion="[[%d,%d],[%d,%d]]" % (c["tn"], c["fp"], c["fn"], c["tp"]), n_organic=int(yc.sum()), n_conventional=int((yc == 0).sum()))

two = ["20220215", "20220216"]; conflicting = dup.index[dup.label_conflict].tolist()
cohorts = {"(a) paper: batches 20220215+20220216, first occurrence (177)": first[first.batch.isin(two)],
           "(b) all four batches, first occurrence": first,
           "(c) all four batches, first occurrence, conflicting identifiers removed": first[~first.echantillon.isin(conflicting)],
           "(d) batches 20220215+20220216, last occurrence of duplicates": raw.drop_duplicates("echantillon", keep="last")[lambda d: d.batch.isin(two)]}
sens = pd.DataFrame([protocol(d, lab) for lab, d in cohorts.items()])
display(sens[["cohort", "n", "cv_accuracy", "cv_sd", "test_accuracy", "test_balanced_accuracy", "test_auc", "confusion"]].round(3))
sens.to_csv(OUT / "milk_batch_sensitivity.csv", index=False, float_format="%.6f")
b = sens.iloc[1]; max_change = max(abs(sens.iloc[2].test_accuracy - sens.iloc[0].test_accuracy), abs(sens.iloc[3].test_accuracy - sens.iloc[0].test_accuracy))

,cohort,n,cv_accuracy,cv_sd,test_accuracy,test_balanced_accuracy,test_auc,confusion
0,"(a) paper: batches 20220215+20220216, first oc...",177,0.922,0.039,0.944,0.889,0.973,"[[27,0],[2,7]]"
1,"(b) all four batches, first occurrence",193,0.915,0.060,0.923,0.875,0.991,"[[27,0],[3,9]]"
2,"(c) all four batches, first occurrence, confli...",192,0.928,0.046,0.897,0.865,0.954,"[[25,1],[3,10]]"
3,"(d) batches 20220215+20220216, last occurrence...",182,0.903,0.048,0.946,0.833,0.930,"[[31,0],[2,4]]"


## 4. Number of descriptors (Figure S1B) and learning curve (Table S27)

The descriptor sweep of the original notebook (panel B of Figure S1): five-fold CV accuracy on the training set as a function of the number
of selected descriptors, from 1 to 199 (selector and forest seed 0). For the learning curve we train the
whole protocol on stratified subsamples of the training set (20 repeats per size, seeds `1000 n + r`) and
score the fixed test set each time.

In [6]:
t0 = time.time(); sizes = list(range(1, 200)); means, sds = [], []
for k in sizes:
    s = cross_val_score(Pipeline([("select", mn.GiniSelector(k, 0)), ("forest", RandomForestClassifier(n_estimators=100, random_state=0))]), X[train], y[train], cv=5)
    means.append(s.mean()); sds.append(s.std())
sweep = pd.DataFrame(dict(n_features=sizes, cv5_mean=means, cv5_sd=sds)); sweep.to_csv(OUT / "figS1B_n_features_sweep.csv", index=False, float_format="%.6f")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, means, color="tab:blue", lw=2, label="Mean CV accuracy"); ax.fill_between(sizes, np.array(means) - sds, np.array(means) + sds, color="tab:blue", alpha=.2, label="±1 STD")
ax.set_xlabel("Number of selected descriptors"); ax.set_ylabel("Mean cross-validation accuracy on training set"); ax.grid(True, ls="--", alpha=.5); ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig(FIG / "figS1B_n_features.pdf"); fig.savefig(FIG / "figS1B_n_features.png", dpi=150); plt.close(fig)
above50 = sweep[sweep.n_features >= 50]
print(f"{time.time() - t0:.0f} s; mean accuracy at 50 / 100 / 150 descriptors: {sweep.cv5_mean[49]:.3f} / {sweep.cv5_mean[99]:.3f} / {sweep.cv5_mean[149]:.3f}; SD at 50 / 100: {sweep.cv5_sd[49]:.3f} / {sweep.cv5_sd[99]:.3f}")

669 s; mean accuracy at 50 / 100 / 150 descriptors: 0.901 / 0.901 / 0.908; SD at 50 / 100: 0.061 / 0.047


In [7]:
rows = []
for n in (20, 30, 40, 60, 80, 100, 120, 141):
    for r in range(20):
        sub = train if n >= len(train) else train[next(StratifiedShuffleSplit(1, train_size=n, random_state=1000 * n + r).split(X[train], y[train]))[0]]
        kk = mn.threshold_selection(X[sub], y[sub], min(100, X.shape[1] - 1), random_state=0)
        m = RandomForestClassifier(n_estimators=100, random_state=2024).fit(X[sub][:, kk], y[sub])
        p = m.predict(X[test][:, kk])
        rows.append(dict(dataset="milk", n_train=len(sub), n_train_pos=int(y[sub].sum()), repeat=r, accuracy=accuracy_score(y[test], p),
                         balanced_accuracy=balanced_accuracy_score(y[test], p), auc=roc_auc_score(y[test], m.predict_proba(X[test][:, kk])[:, 1])))
        if n >= len(train):
            break
lc_rep = pd.DataFrame(rows)
lc = lc_rep.groupby("n_train").agg(n_train_pos=("n_train_pos", "first"), repeats=("repeat", "count"), accuracy_mean=("accuracy", "mean"), accuracy_sd=("accuracy", "std"),
                                   accuracy_min=("accuracy", "min"), balanced_mean=("balanced_accuracy", "mean"), auc_mean=("auc", "mean"), auc_sd=("auc", "std")).reset_index()
lc.insert(0, "dataset", "milk"); display(lc.round(3))
lc_rep.to_csv(OUT / "learning_curve_repeats.csv", index=False, float_format="%.6f"); lc.to_csv(OUT / "learning_curve_summary.csv", index=False, float_format="%.6f")

,dataset,n_train,n_train_pos,repeats,accuracy_mean,accuracy_sd,accuracy_min,balanced_mean,auc_mean,auc_sd
0,milk,20,6,20,0.862,0.035,0.750,0.792,0.883,0.031
1,milk,30,8,20,0.869,0.030,0.806,0.796,0.900,0.036
2,milk,40,11,20,0.876,0.032,0.806,0.808,0.910,0.039
3,milk,60,17,20,0.883,0.025,0.861,0.830,0.939,0.025
4,milk,80,22,20,0.889,0.020,0.861,0.837,0.936,0.028
5,milk,100,28,20,0.912,0.027,0.861,0.866,0.957,0.015
6,milk,120,33,20,0.901,0.029,0.861,0.855,0.960,0.012
7,milk,141,39,1,0.944,NaN,0.944,0.889,0.973,NaN


## 5. Reading the bins

Once the model is trained, the bins are ranked by the Gini importance of their descriptors and we look at
them one at a time through the stored m/z and RT of the retained maxima. Bin 183 (m/z [130,140), RT [6,8)
min) is the worked example of the text (Figure 2, Figure S3A); the other regions of Table 2 are bins 278,
121 and 192 (bin 121, a crowded window with nine m/z sub-clusters, is panel B of Figure S3; bins 278 and
192 are no longer shown in the Supporting Information). The ranks are those of the selector forest (seed 0, training set, 1 260
descriptors) and of the final model (101 descriptors); the counts cover the 177-sample cohort.

In [8]:
imp_sel = RandomForestClassifier(n_estimators=100, random_state=0).fit(X[train], y[train]).feature_importances_
imp_final = np.zeros(X.shape[1]); imp_final[np.flatnonzero(keep)] = model.feature_importances_
print("ten most important descriptors of the selector forest:", [f"{'mz' if i < NB else 'RT'}[bin {i % NB}]" for i in np.argsort(imp_sel)[::-1][:10]])

def group_stats(b, lo, hi, label):
    co = mn.bin_coordinates(X, b, GRID, components=("mz", "rt")); sel = co.mz.between(lo, hi).to_numpy()
    conv, org = int((y[sel] == 0).sum()), int((y[sel] == 1).sum())
    p = fisher_exact([[conv, int((y == 0).sum()) - conv], [org, int((y == 1).sum()) - org]])[1]
    mz = co.mz[sel]
    return dict(bin=b, bin_limits=mn.bin_label(b, GRID), group=label, mz_selection=f"[{lo}, {hi}]", gini_rank_selector_mz=mn.rank_of(imp_sel, b), gini_rank_selector_rt=mn.rank_of(imp_sel, b + NB),
                gini_rank_final_model_mz=mn.rank_of(imp_final, b) if keep[b] else None, n=int(sel.sum()), conventional=conv, organic=org, fisher_p=p,
                mean_mz=mz.mean(), sd_mz=mz.std(ddof=0), sd_mz_ppm=mz.std(ddof=0) / mz.mean() * 1e6, mean_rt=co.rt[sel].mean(), sd_rt=co.rt[sel].std(ddof=0))

regions = [group_stats(183, 138.0, 139.0, "HexNAc oxonium fragment C7H8NO2+ (138.055)"), group_stats(183, 132.09, 132.12, "Leu/Ile [M+H]+ (132.102)"),
           group_stats(278, 184.08, 184.11, "carnitine [M+Na]+ (184.094)"), group_stats(278, 189.11, 189.14, "C8H16N2O3 [M+H]+ (189.123)"),
           group_stats(121, 412.40, 412.43, "412.415 (C26:1 [M+NH4]+ or C26H53NO2 [M+H]+)"), group_stats(192, 222.08, 222.11, "HexNAc [M+H]+ (222.097)")]
table2 = pd.DataFrame(regions); display(table2.drop(columns=["bin"]).round(5)); table2.to_csv(OUT / "table2_regions.csv", index=False, float_format="%.8g")
r183 = regions[0]
print(f"bin 183: {r183['n']} maxima at 138-139 ({r183['conventional']} conventional, {r183['organic']} organic), mean m/z {r183['mean_mz']:.5f} (SD {r183['sd_mz']:.5f} = {r183['sd_mz_ppm']:.1f} ppm), RT {r183['mean_rt']:.2f} ± {r183['sd_rt']:.2f} min, Fisher p = {r183['fisher_p']:.3e}")
co183 = mn.bin_coordinates(X, 183, GRID, components=("mz", "rt")); sel_t = co183.mz[test].between(138, 139).to_numpy()
ct, ot = int((y[test][sel_t] == 0).sum()), int((y[test][sel_t] == 1).sum()); p_t = fisher_exact([[ct, 27 - ct], [ot, 9 - ot]])[1]
print(f"test set: {ct} conventional and {ot} organic in the 138-139 group, Fisher p = {p_t:.3f}")
expected = {"HexNAc oxonium": (21, "105/8", 0.8), "Leu/Ile": (21, "22/39", 0.7), "carnitine": (6, "110/10", 0.7), "C8H16N2O3": (6, "8/27", 0.6),
            "412.415": (30, "55/5", 0.7), "HexNAc [M+H]": (51, "82/4", 0.8)}        # Table 2: Gini rank, conv/org counts, cluster SD (ppm)
for r in regions:
    key = next(k for k in expected if r["group"].startswith(k)); rank_exp, counts_exp, ppm_exp = expected[key]

ten most important descriptors of the selector forest: ['RT[bin 175]', 'RT[bin 197]', 'RT[bin 222]', 'mz[bin 216]', 'RT[bin 142]', 'mz[bin 278]', 'mz[bin 201]', 'mz[bin 343]', 'RT[bin 160]', 'RT[bin 128]']


,bin_limits,group,mz_selection,gini_rank_selector_mz,gini_rank_selector_rt,gini_rank_final_model_mz,n,conventional,organic,fisher_p,mean_mz,sd_mz,sd_mz_ppm,mean_rt,sd_rt
0,"m/z [130,140) RT [6,8) min",HexNAc oxonium fragment C7H8NO2+ (138.055),"[138.0, 139.0]",21,549,20,113,105,8,0.00000,138.05474,0.00011,0.81042,6.48942,0.14566
1,"m/z [130,140) RT [6,8) min",Leu/Ile [M+H]+ (132.102),"[132.09, 132.12]",21,549,20,61,22,39,0.00000,132.10167,0.00010,0.72449,7.18486,0.03042
2,"m/z [180,190) RT [8,10) min",carnitine [M+Na]+ (184.094),"[184.08, 184.11]",6,15,4,120,110,10,0.00000,184.09401,0.00013,0.72960,9.01714,0.03063
3,"m/z [180,190) RT [8,10) min",C8H16N2O3 [M+H]+ (189.123),"[189.11, 189.14]",6,15,4,35,8,27,0.00000,189.12304,0.00011,0.60687,8.28422,0.03206
4,"m/z [410,420) RT [4,6) min",412.415 (C26:1 [M+NH4]+ or C26H53NO2 [M+H]+),"[412.4, 412.43]",30,549,74,60,55,5,0.00003,412.41410,0.00029,0.70551,5.18141,0.03306
5,"m/z [220,230) RT [6,8) min",HexNAc [M+H]+ (222.097),"[222.08, 222.11]",51,520,95,86,82,4,0.00000,222.09685,0.00018,0.79587,6.48104,0.02029


bin 183: 113 maxima at 138-139 (105 conventional, 8 organic), mean m/z 138.05474 (SD 0.00011 = 0.8 ppm), RT 6.49 ± 0.15 min, Fisher p = 2.365e-15
test set: 23 conventional and 3 organic in the 138-139 group, Fisher p = 0.006


How stable is the rank of bin 183? We refit the selector forest with ten seeds on the training set, and
look at the in-fold selection of the ten training folds (top-100 Gini, seed 0).

In [9]:
ranks = [mn.rank_of(RandomForestClassifier(n_estimators=100, random_state=s).fit(X[train], y[train]).feature_importances_, 183) for s in range(10)]
hits = 0; fold_ranks = []
for tr, _ in StratifiedKFold(10, shuffle=False).split(X[train], y[train]):
    imp = RandomForestClassifier(n_estimators=100, random_state=0).fit(X[train][tr], y[train][tr]).feature_importances_
    r = mn.rank_of(imp, 183); fold_ranks.append(r); hits += r <= 100
print(f"rank of mz[bin 183] over seeds 0-9: {ranks} -> median {np.median(ranks):.0f}, min {min(ranks)}, max {max(ranks)}; selected (top 100) in {hits}/10 training folds (per-fold ranks {fold_ranks})")
pd.DataFrame(dict(seed=range(10), rank_mz_bin183=ranks)).to_csv(OUT / "bin183_rank_over_seeds.csv", index=False)

rank of mz[bin 183] over seeds 0-9: [21, 21, 1, 45, 30, 43, 530, 48, 159, 104] -> median 44, min 1, max 530; selected (top 100) in 8/10 training folds (per-fold ranks [519, 11, 17, 18, 49, 15, 14, 38, 311, 20])


In [10]:
# Figure 2 (m/z vs RT, test set) and Figure S3A (m/z histogram, test set) of bin 183; Figure S3B: bin 121 over the 177 samples; bins 278 and 192 are drawn the same way but are not in the Supporting Information (files nb_bin278, nb_bin192)
def scatter(ax, b, idx, xlim=None, ylim=None, title=""):
    co = mn.bin_coordinates(X, b, GRID, components=("mz", "rt"))
    for lab, col, name in ((0, "tab:blue", "Conventional"), (1, "tab:orange", "Organic")):
        s = idx[y[idx] == lab]; ax.scatter(co.mz[s], co.rt[s], c=col, edgecolor="k", s=60, alpha=.8, label=name)
    ax.set_xlabel("m/z"); ax.set_ylabel("Retention time (minute)"); ax.grid(True, ls="--", alpha=.5); ax.legend(title="Category")
    if xlim: ax.set_xlim(*xlim)
    if ylim: ax.set_ylim(*ylim)
    ax.set_title(title, fontsize=9)
fig, ax = plt.subplots(figsize=(6.5, 4.8)); scatter(ax, 183, test, title="Figure 2: bin 183 (m/z [130,140), RT [6,8) min), test set n = 36")
plt.tight_layout(); fig.savefig(FIG / "fig2_bin183_test.pdf"); fig.savefig(FIG / "fig2_bin183_test.png", dpi=150); plt.close(fig)
fig, ax = plt.subplots(figsize=(6.5, 4)); co = co183
for lab, col, name in ((0, "tab:blue", "Conventional"), (1, "tab:orange", "Organic")):
    s = test[y[test] == lab]; ax.hist(co.mz[s], bins=np.linspace(130, 140, 51), color=col, alpha=.7, label=name)
ax.invert_yaxis(); ax.set_xlabel("m/z"); ax.set_ylabel("Frequency"); ax.legend(); ax.set_title("Figure S3A: m/z of the retained maxima of bin 183, test set", fontsize=9)
plt.tight_layout(); fig.savefig(FIG / "figS3A_bin183_hist_test.pdf"); fig.savefig(FIG / "figS3A_bin183_hist_test.png", dpi=150); plt.close(fig)
all_idx = np.arange(len(y))
for b, name, xl, yl in ((278, "nb_bin278", (181, 190), (8, 10)), (121, "figS3B_bin121", (410, 420), (4, 6)), (192, "nb_bin192", (220, 230), (6.4, 7.6))):
    fig, ax = plt.subplots(figsize=(6.5, 4.8)); scatter(ax, b, all_idx, xl, yl, f"bin {b} ({mn.bin_label(b, GRID)}), 177 samples")
    plt.tight_layout(); fig.savefig(FIG / f"{name}.pdf"); fig.savefig(FIG / f"{name}.png", dpi=150); plt.close(fig)
print("figures written to", os.path.relpath(FIG))

figures written to results\milk\figures


## 6. Downstream inspection of the ten most important bins

For the ten bins with the largest Gini importance (maximum over their two descriptors), plus bin 183:
occupancy, dispersion of the retained maxima, dominant coordinate group (full span 10 ppm × 30 s; support
*D* = group / 177, consistency *C* = group / present), DBSCAN m/z sub-clusters and a one-line flag.

In [11]:
per_bin = imp_sel.reshape(2, NB).max(axis=0); order = np.argsort(per_bin)[::-1][:10]
rows = []
for b in list(order) + [183]:
    r = mn.inspect_bin(mn.bin_coordinates(X, b, GRID, components=("mz", "rt")), y, int(b), GRID, ("conv", "org"), 10e-6, 30.0)
    r.update(importance=float(per_bin[b]), rank=int((per_bin > per_bin[b]).sum()) + 1); rows.append(r)
insp = pd.DataFrame(rows); insp.to_csv(OUT / "downstream_inspection_milk.csv", index=False, float_format="%.6g")
display(insp[["rank", "bin", "bin_limits", "occupancy", "mz_sd_ppm", "rt_sd_min", "support_D", "consistency_C", "n_mz_subclusters", "class_counts_present", "flag"]].round(3))
single = insp.iloc[:10][insp.iloc[:10].consistency_C >= 0.9]

,rank,bin,bin_limits,occupancy,mz_sd_ppm,rt_sd_min,support_D,consistency_C,n_mz_subclusters,class_counts_present,flag
0,1,175,"m/z [950,960) RT [4,6) min",1.0,1053.571,0.111,0.960,0.960,2,129 conv/48 org,single dominant ion: proceed to XIC / isotopes...
1,2,197,"m/z [270,280) RT [6,8) min",1.0,10398.156,0.492,0.548,0.548,3,129 conv/48 org,mixed column: sub-cluster before any chemical ...
2,3,222,"m/z [520,530) RT [6,8) min",1.0,993.009,0.303,0.927,0.927,2,129 conv/48 org,single dominant ion: proceed to XIC / isotopes...
3,4,216,"m/z [460,470) RT [6,8) min",1.0,4905.831,0.197,0.672,0.672,3,129 conv/48 org,mixed column: sub-cluster before any chemical ...
4,5,142,"m/z [620,630) RT [4,6) min",1.0,3292.546,0.515,0.870,0.870,4,129 conv/48 org,mixed column: sub-cluster before any chemical ...
5,6,278,"m/z [180,190) RT [8,10) min",1.0,11662.092,0.365,0.678,0.678,4,129 conv/48 org,mixed column: sub-cluster before any chemical ...
6,7,201,"m/z [310,320) RT [6,8) min",1.0,7138.055,0.289,0.582,0.582,5,129 conv/48 org,mixed column: sub-cluster before any chemical ...
7,8,343,"m/z [830,840) RT [8,10) min",1.0,2571.590,0.331,0.684,0.684,5,129 conv/48 org,mixed column: sub-cluster before any chemical ...
8,9,160,"m/z [800,810) RT [4,6) min",1.0,428.002,0.136,0.898,0.898,2,129 conv/48 org,mixed column: sub-cluster before any chemical ...
9,10,128,"m/z [480,490) RT [4,6) min",1.0,327.172,0.152,0.994,0.994,1,129 conv/48 org,single dominant ion: proceed to XIC / isotopes...


## 7. Feature-level coordinate check of the deposited table (Supporting Information, section 5.2)

For each of the 630 bins we count the largest number of the 177 retained maxima that fit in a rectangle
of full span 10 ppm × 30 s (and 5 ppm × 15 s), divided by 177: the joint support *D* of the dominant
coordinate group. The article reports how many bins reach *D* ≥ 0.9.

In [12]:
def support_table(ppm, rt_s):
    out = []
    for b in range(NB):
        co = mn.bin_coordinates(X, b, GRID, components=("mz", "rt")); present = (X[:, b] > 0) | (X[:, NB + b] > 0)
        G = mn.dominant_group(co.mz.to_numpy()[present], co.rt.to_numpy()[present], ppm, rt_s) if present.any() else 0
        out.append(dict(bin=b, n_present=int(present.sum()), dominant=G, D=G / len(y)))
    return pd.DataFrame(out)
t0 = time.time(); D10 = support_table(10e-6, 30.0); D5 = support_table(5e-6, 15.0)
viol = sum(not mn.bin_coordinates(X, b, GRID, components=("mz", "rt")).mz[(X[:, b] > 0)].between(*mn.bin_limits(b, GRID)[:2], inclusive="left").all() for b in range(NB))
print(f"{time.time() - t0:.0f} s; bins with D >= 0.9: {int((D10.D >= 0.9).sum())}/630 at 10 ppm x 30 s, {int((D5.D >= 0.9).sum())}/630 at 5 ppm x 15 s; bin-boundary violations {viol}")
D10.to_csv(OUT / "section5_2_support_10ppm_30s.csv", index=False); D5.to_csv(OUT / "section5_2_support_5ppm_15s.csv", index=False)

388 s; bins with D >= 0.9: 112/630 at 10 ppm x 30 s, 89/630 at 5 ppm x 15 s; bin-boundary violations 0


## 8. Raw data: the named ions in the 166 available injections

One pass per mzXML file (centroided MS$^1$). For every ion each candidate should produce, we extract the ion
chromatogram (±5 ppm): the N-acetylhexosamine family ([M+H]⁺ 222.097, [M+Na]⁺ 244.079, the oxonium ladder
204/186/168/144/138/126) and the two PABA-specific ions, the carnitine adducts and fragments, the
companions of the 412.415 ion, the Leu/Ile [M+H]⁺ and the 189.123 ion. We record the isotope ratios at
the apex, the peak areas, the retained maximum of the four MINUT bins of Table 2 and the normalizers of the run.
The summaries are Tables S8–S10 of the Supporting Information (ion ladders and XIC correlations;
conventional/organic ratios pooled and within each acquisition batch; retained maxima by category), together
with the isotope ratios and chromatographic positions discussed in its section 5.4; the distribution of the
189.123 ion over sample identifiers is written to the results only.

In [13]:
from minut import H, H2O, K, NH4, Na, D13C, iso_expect, peak, read_scans, bin_maximum
HEX, CAR, C26, LEU, ACK, PABA = 221.08994, 161.10519, 394.38108, 131.09463, 188.11609, 137.04768   # neutral masses
IONS = {  # name: (m/z, RT bin of the apex search)
    "hex_222": (HEX + H, (5.5, 7.5)), "hex_244Na": (HEX + Na, (5.5, 7.5)), "hex_204": (204.08665, (5.5, 7.5)), "hex_186": (186.07608, (5.5, 7.5)),
    "hex_168": (168.06552, (5.5, 7.5)), "hex_144": (144.06552, (5.5, 7.5)), "hex_138": (138.05495, (5.5, 7.5)), "hex_126": (126.05495, (5.5, 7.5)),
    "paba_120": (120.04439, (5.5, 7.5)), "paba_160Na": (PABA + Na, (5.5, 7.5)),
    "car_162H": (CAR + H, (7.5, 10.5)), "car_184Na": (CAR + Na, (7.5, 10.5)), "car_200K": (CAR + K, (7.5, 10.5)),
    "car_103frag": (CAR + H - 59.07350, (7.5, 10.5)), "car_144H2O": (CAR + H - H2O, (7.5, 10.5)),
    "c26_412NH4": (C26 + NH4, (3.5, 6.5)), "c26_395H": (C26 + H, (3.5, 6.5)), "c26_417Na": (C26 + Na, (3.5, 6.5)),
    "c26_377H2O": (C26 + H - H2O, (3.5, 6.5)), "c26_433K": (C26 + K, (3.5, 6.5)),
    "leu_132H": (LEU + H, (6.5, 8.0)), "leu_154Na": (LEU + Na, (6.5, 8.0)),
    "ack_189H": (ACK + H, (7.5, 9.0)), "ack_211Na": (ACK + Na, (7.5, 9.0)), "ack_147": (147.11280, (7.5, 9.0)), "ack_130": (130.08626, (7.5, 9.0)),
}
ISO = {"hex_222": "C8H15NO6", "hex_138": "C7H7NO2", "car_162H": "C7H15NO3", "car_184Na": "C7H15NO3", "c26_412NH4": "C26H50O2", "leu_132H": "C6H13NO2", "ack_189H": "C8H16N2O3"}
BINS = {183: ((130, 140), (6, 8)), 192: ((220, 230), (6, 8)), 278: ((180, 190), (8, 10)), 121: ((410, 420), (4, 6))}
PEAKS = {"hex": ("hex_204", ["hex_222", "hex_244Na", "hex_186", "hex_168", "hex_144", "hex_138", "hex_126", "paba_120", "paba_160Na"], (4.0, 9.0)),
         "car": ("car_162H", ["car_184Na", "car_200K", "car_103frag", "car_144H2O"], (7.5, 10.5)),
         "c26": ("c26_412NH4", ["c26_395H", "c26_417Na", "c26_377H2O", "c26_433K"], (3.5, 6.5)),
         "leu": ("leu_132H", ["leu_154Na"], (6.5, 8.0)),
         "ack": ("ack_189H", ["ack_211Na", "ack_147", "ack_130"], (7.5, 9.0))}

def process(path):
    path = Path(path); parts = path.stem.split("_")
    rec = dict(file=path.stem, category=parts[1], sample_id=parts[2], batch=parts[-1])
    rows = []; iso = {k: [] for k in ISO}; win = {b: (0.0, np.nan, np.nan) for b in BINS}
    for rt, mz, it in read_scans(path):
        row = dict(rt=rt, tic=it.sum(), bpi=it.max() if it.size else 0.0)
        for k, (m, _) in IONS.items():
            row[k] = peak(mz, it, m)
        for k in ISO:
            m = IONS[k][0]
            iso[k].append((peak(mz, it, m + D13C), peak(mz, it, m + 2 * D13C), peak(mz, it, m + D13C / 2)) if row[k] > 0 else (0.0, 0.0, 0.0))
        for b, ((mlo, mhi), (rlo, rhi)) in BINS.items():
            if rlo <= rt < rhi:
                i, m = bin_maximum(mz, it, mlo, mhi)
                if i > win[b][0]:
                    win[b] = (i, m, rt)
        rows.append(row)
    t = pd.DataFrame(rows)
    rec.update(n_scans=len(t), tic_sum=t.tic.sum(), bpi_max=t.bpi.max())
    for b, (i, m, rt) in win.items():
        rec[f"bin{b}_retained_mz"] = round(m, 4) if m == m else np.nan; rec[f"bin{b}_retained_rt"] = round(rt, 2) if rt == rt else np.nan; rec[f"bin{b}_retained_I"] = i
    for tag, (ref, comps, (lo, hi)) in PEAKS.items():
        w = t[(t.rt >= IONS[ref][1][0]) & (t.rt < IONS[ref][1][1])]
        if w[ref].max() <= 0:
            continue
        a = w[ref].idxmax(); apex_rt = t.rt[a]; pk = t[(t.rt >= apex_rt - 0.15) & (t.rt <= apex_rt + 0.15)]
        rec[f"{tag}_apex_rt"] = round(apex_rt, 3); rec[f"{tag}_apex_I"] = pk[ref].max(); rec[f"{tag}_area"] = pk[ref].sum()
        if tag == "hex":   # both anomers (second peak ~0.15 min later) for the HexNAc areas
            both = t[(t.rt >= apex_rt - 0.2) & (t.rt <= apex_rt + 0.5)]
            rec["hex_area_204_both"] = both.hex_204.sum(); rec["hex_area_222_both"] = both.hex_222.sum(); rec["hex_area_138_both"] = both.hex_138.sum()
            sec = t[(t.rt > apex_rt + 0.15) & (t.rt <= apex_rt + 0.5)]
            rec["hex_second_anomer_rt"] = round(t.rt[sec.hex_204.idxmax()], 2) if len(sec) and sec.hex_204.max() > 0.1 * pk.hex_204.max() else np.nan
            out = t[(t.rt < apex_rt - 0.3) | (t.rt > apex_rt + 0.8)]   # any 138.055 elsewhere not explained by HexNAc?
            resid = (out.hex_138 - 0.5 * out.hex_204).clip(lower=0)
            rec["other138_residual_frac"] = round(resid.max() / max(t.hex_138.max(), 1), 3); rec["other138_residual_rt"] = round(t.rt[resid.idxmax()], 2)
        ww = t[(t.rt >= lo) & (t.rt < hi)]
        for c in comps:
            rec[f"{c}/{ref}"] = round(pk[c].sum() / max(pk[ref].sum(), 1), 4)
            rec[f"corr_{c}"] = round(np.corrcoef(ww[c], ww[ref])[0, 1], 3) if ww[c].std() > 0 and ww[ref].std() > 0 else np.nan
    for k, f in ISO.items():
        lo, hi = IONS[k][1]; m1 = np.array([v[0] for v in iso[k]]); m2 = np.array([v[1] for v in iso[k]]); hf = np.array([v[2] for v in iso[k]])
        sel = (t.rt >= lo) & (t.rt < hi) & (t[k] > 0)
        if sel.any():
            top = sel & (t[k] >= 0.5 * t[k][sel].max())
            rec[f"{k}_M1/M"] = round(m1[top].sum() / t[k][top].sum(), 4); rec[f"{k}_M2/M"] = round(m2[top].sum() / t[k][top].sum(), 4); rec[f"{k}_half/M"] = round(hf[top].sum() / t[k][top].sum(), 4)
    return rec

per_sample = OUT / "milk_annotations_per_sample.csv"
d = mn.cached(per_sample)
if d is None and mn.raw_available("milk"):
    files = sorted(str(p) for p in (mn.RAW / "milk").rglob("*.mzXML")); t0 = time.time()
    print(f"raw pass on {len(files)} mzXML files...", flush=True)
    d = pd.DataFrame(Parallel(n_jobs=6)(delayed(process)(f) for f in files)).sort_values("file")
    d.to_csv(per_sample, index=False); print(f"{time.time() - t0:.0f} s")
    d = mn.cached(per_sample)
if d is None:
    raise RuntimeError("no raw milk files and no per-sample table: set MINUT_RAW or copy results/milk/milk_annotations_per_sample.csv")
print(f"{len(d)} injections: {(d.category == 'b').sum()} organic, {(d.category == 'c').sum()} conventional; batches {d.batch.value_counts().to_dict()}")

raw pass on 166 mzXML files...


899 s
166 injections: 44 organic, 122 conventional; batches {'20220216': 86, '20220215': 67, '20220214': 13}


In [14]:
b_, c_ = d[d.category == "b"], d[d.category == "c"]
# 1. ion ladders: ratio to the reference ion and XIC correlation (Table S8)
rows = []
for tag, (ref, comps, _) in PEAKS.items():
    for comp in comps:
        v = d[f"{comp}/{ref}"]; cc = d[f"corr_{comp}"]
        rows.append(dict(candidate=tag, reference=ref, ion=comp, mz=round(IONS[comp][0], 4), ratio_median=v.median(), ratio_q25=v.quantile(.25), ratio_q75=v.quantile(.75),
                         ratio_b=b_[f"{comp}/{ref}"].median(), ratio_c=c_[f"{comp}/{ref}"].median(), corr_median=cc.median(), corr_min=cc.min(), detected_frac=(v > 0).mean()))
ladders = pd.DataFrame(rows); ladders.to_csv(OUT / "milk_ion_ladders.csv", index=False); display(ladders.round(3))
# 2. isotopes (SI section 5.4, text)
iso_t = pd.DataFrame([dict(ion=k, formula=f, M1_obs=d[f"{k}_M1/M"].median(), M1_q25=d[f"{k}_M1/M"].quantile(.25), M1_q75=d[f"{k}_M1/M"].quantile(.75), M1_exp=iso_expect(f, 1),
                           M2_obs=d[f"{k}_M2/M"].median(), M2_exp=iso_expect(f, 2), half_obs=d[f"{k}_half/M"].median(), n=d[f"{k}_M1/M"].notna().sum()) for k, f in ISO.items()])
iso_t.to_csv(OUT / "milk_isotopes.csv", index=False); display(iso_t.round(4))
# 3. detection / chromatography (SI section 5.4, text)
det = pd.DataFrame([dict(candidate=tag, detected=d[f"{tag}_apex_rt"].notna().sum(), n=len(d), rt_mean=d[f"{tag}_apex_rt"].mean(), rt_sd=d[f"{tag}_apex_rt"].std(), rt_min=d[f"{tag}_apex_rt"].min(), rt_max=d[f"{tag}_apex_rt"].max()) for tag in PEAKS]
                   + [dict(candidate="hex second anomer", detected=d.hex_second_anomer_rt.notna().sum(), n=len(d), rt_mean=d.hex_second_anomer_rt.mean(), rt_sd=d.hex_second_anomer_rt.std(), rt_min=d.hex_second_anomer_rt.min(), rt_max=d.hex_second_anomer_rt.max())])
det.to_csv(OUT / "milk_detection.csv", index=False); display(det.round(3))

,candidate,reference,ion,mz,ratio_median,ratio_q25,ratio_q75,ratio_b,ratio_c,corr_median,corr_min,detected_frac
0,hex,hex_204,hex_222,222.097,0.150,0.135,0.162,0.133,0.157,0.992,0.813,1.000
1,hex,hex_204,hex_244Na,244.079,0.146,0.120,0.218,0.246,0.136,0.907,0.774,1.000
2,hex,hex_204,hex_186,186.076,0.756,0.720,0.864,0.869,0.738,0.996,0.821,1.000
3,hex,hex_204,hex_168,168.066,0.204,0.138,0.212,0.132,0.207,0.967,0.356,1.000
4,hex,hex_204,hex_144,144.066,0.175,0.155,0.246,0.250,0.167,0.973,0.833,1.000
5,hex,hex_204,hex_138,138.055,0.356,0.285,0.376,0.266,0.365,0.987,0.619,1.000
6,hex,hex_204,hex_126,126.055,0.154,0.139,0.183,0.185,0.147,0.978,0.918,1.000
7,hex,hex_204,paba_120,120.044,0.000,0.000,0.000,0.000,0.000,-0.172,-0.238,0.000
8,hex,hex_204,paba_160Na,160.037,0.000,0.000,0.000,0.000,0.000,-0.014,-0.069,0.000
9,car,car_162H,car_184Na,184.094,0.002,0.002,0.003,0.002,0.002,0.938,0.021,1.000


,ion,formula,M1_obs,M1_q25,M1_q75,M1_exp,M2_obs,M2_exp,half_obs,n
0,hex_222,C8H15NO6,0.0755,0.0746,0.0766,0.0902,0.0001,0.0159,0.0,166
1,hex_138,C7H7NO2,0.0650,0.0632,0.0661,0.0794,0.0014,0.0068,0.0,166
2,car_162H,C7H15NO3,0.0765,0.0760,0.0768,0.0794,0.0015,0.0089,0.0,166
3,car_184Na,C7H15NO3,0.0600,0.0475,0.0683,0.0794,0.0000,0.0089,0.0,166
4,c26_412NH4,C26H50O2,0.2689,0.2610,0.2777,0.2812,0.0267,0.0421,0.0,166
5,leu_132H,C6H13NO2,0.0545,0.0529,0.0562,0.0685,0.0005,0.0061,0.0,166
6,ack_189H,C8H16N2O3,0.0250,0.0000,0.0716,0.0938,0.0000,0.0101,0.0,166


,candidate,detected,n,rt_mean,rt_sd,rt_min,rt_max
0,hex,166,166,6.442,0.064,6.239,6.590
1,car,166,166,8.989,0.028,8.918,9.042
2,c26,166,166,5.177,0.049,5.013,5.453
3,leu,166,166,7.213,0.033,7.154,7.456
4,ack,166,166,8.350,0.282,7.518,8.998
5,hex second anomer,166,166,6.599,0.066,6.390,6.740


In [15]:
# 4. conventional/organic ratios of the XIC areas, pooled and within each of the two large batches (Table S9)
rows = []
for key, lab in [("hex_area_204_both", "HexNAc 204.087 (oxonium) area"), ("hex_area_222_both", "HexNAc [M+H]+ 222.097 area"), ("hex_area_138_both", "HexNAc fragment 138.055 area"),
                 ("car_area", "carnitine [M+H]+ 162.112 area"), ("c26_area", "412.415 area"), ("leu_area", "Leu/Ile [M+H]+ 132.102 area"), ("ack_area", "189.123 area")]:
    for norm, nl in [(None, "raw"), ("tic_sum", "TIC-normalized")]:
        vb = b_[key] / (b_[norm] if norm else 1); vc = c_[key] / (c_[norm] if norm else 1)
        row = dict(ion=lab, normalization=nl, pooled_conv_over_org=vc.median() / vb.median(), pooled_p=mn.mwu(vb, vc))
        for batch in ["20220215", "20220216"]:
            lb, lc_ = b_[b_.batch == batch], c_[c_.batch == batch]
            xb = lb[key] / (lb[norm] if norm else 1); xc = lc_[key] / (lc_[norm] if norm else 1)
            row[f"batch{batch[-2:]}_conv_over_org"] = xc.median() / xb.median(); row[f"batch{batch[-2:]}_p"] = mn.mwu(xb, xc); row[f"batch{batch[-2:]}_n_org_conv"] = f"{len(lb)}/{len(lc_)}"
        rows.append(row)
ratios = pd.DataFrame(rows); ratios.to_csv(OUT / "milk_category_ratios.csv", index=False); display(ratios.round(4))
hx = ratios[ratios.ion.str.startswith("HexNAc") & ~ratios.ion.str.contains("138")]      # the two HexNAc ions of the text (204, 222), raw and TIC-normalized
leu = ratios[ratios.ion.str.startswith("Leu")]; car = ratios[ratios.ion.str.startswith("carnitine")]
print(f"HexNAc 204/222, batch 20220215: x{hx.batch15_conv_over_org.min():.2f}-{hx.batch15_conv_over_org.max():.2f} (p <= {hx.batch15_p.max():.1e}); batch 20220216: x{hx.batch16_conv_over_org.min():.2f}-{hx.batch16_conv_over_org.max():.2f} (p <= {hx.batch16_p.max():.1e})")
print(f"Leu/Ile organic/conventional: batch 15 x{1 / leu.batch15_conv_over_org.max():.2f}-{1 / leu.batch15_conv_over_org.min():.2f} (p <= {leu.batch15_p.max():.1e}); batch 16 x{1 / leu.batch16_conv_over_org.max():.2f}-{1 / leu.batch16_conv_over_org.min():.2f} (p <= {leu.batch16_p.max():.1e})")
print(f"carnitine conventional/organic within batches: x{min(car.batch15_conv_over_org.min(), car.batch16_conv_over_org.min()):.2f}-{max(car.batch15_conv_over_org.max(), car.batch16_conv_over_org.max()):.2f}")
pe = lambda p: f"{p:.0e}".replace("e-0", "e-")

,ion,normalization,pooled_conv_over_org,pooled_p,batch15_conv_over_org,batch15_p,batch15_n_org_conv,batch16_conv_over_org,batch16_p,batch16_n_org_conv
0,HexNAc 204.087 (oxonium) area,raw,1.8456,0.0000,1.6877,0.0006,13/54,2.0028,0.0000,24/62
1,HexNAc 204.087 (oxonium) area,TIC-normalized,2.1814,0.0000,2.2459,0.0000,13/54,2.3395,0.0000,24/62
2,HexNAc [M+H]+ 222.097 area,raw,2.2917,0.0000,2.0427,0.0001,13/54,2.1909,0.0000,24/62
3,HexNAc [M+H]+ 222.097 area,TIC-normalized,2.7634,0.0000,2.8117,0.0000,13/54,2.5866,0.0000,24/62
4,HexNAc fragment 138.055 area,raw,2.8430,0.0000,2.6223,0.0011,13/54,3.2742,0.0000,24/62
5,HexNAc fragment 138.055 area,TIC-normalized,3.3886,0.0000,3.6152,0.0000,13/54,3.6380,0.0000,24/62
6,carnitine [M+H]+ 162.112 area,raw,1.2235,0.0003,1.2257,0.0091,13/54,1.2418,0.0062,24/62
7,carnitine [M+H]+ 162.112 area,TIC-normalized,1.4294,0.0000,1.6410,0.0003,13/54,1.3920,0.0003,24/62
8,412.415 area,raw,1.5746,0.0000,2.1531,0.0000,13/54,1.2790,0.0121,24/62
9,412.415 area,TIC-normalized,1.9329,0.0000,2.5096,0.0000,13/54,1.4775,0.0044,24/62


HexNAc 204/222, batch 20220215: x1.69-2.81 (p <= 6.0e-04); batch 20220216: x2.00-2.59 (p <= 1.1e-09)
Leu/Ile organic/conventional: batch 15 x2.60-2.77 (p <= 5.0e-04); batch 16 x2.67-3.24 (p <= 6.1e-08)
carnitine conventional/organic within batches: x1.23-1.64


In [16]:
# 5. retained maxima by category (Table S10) and 6. the 189.123 ion over sample identifiers (results only)
rows = []
for bn, ((mlo, mhi), (rlo, rhi)) in BINS.items():
    w = d[f"bin{bn}_retained_mz"].round(2)
    for m, g in d.groupby(w):
        if len(g) < 2:
            continue
        rows.append(dict(bin=bn, bin_limits=f"m/z [{mlo},{mhi}) x RT [{rlo},{rhi})", retained_mz=m, retained_mz_mean=g[f"bin{bn}_retained_mz"].mean(), n_org=(g.category == "b").sum(), n_conv=(g.category == "c").sum(),
                         rt_median=g[f"bin{bn}_retained_rt"].median(), I_median_org=g[g.category == "b"][f"bin{bn}_retained_I"].median(), I_median_conv=g[g.category == "c"][f"bin{bn}_retained_I"].median()))
retained = pd.DataFrame(rows).sort_values(["bin", "n_conv"], ascending=[True, False]); retained.to_csv(OUT / "milk_bin_retained_maxima.csv", index=False); display(retained.round(3))
d2 = d.copy(); d2["log189"] = np.log10(d2.ack_area.fillna(0).clip(lower=1)); d2["present189"] = d2.log189 > 6.5; d2["sid"] = d2.sample_id.astype(int)
blocks = d2.groupby(["category", "batch"]).agg(n=("present189", "size"), present=("present189", "sum")).reset_index(); blocks.to_csv(OUT / "milk_189_by_batch.csv", index=False)
d2.sort_values(["category", "sid"])[["file", "category", "batch", "sid", "log189", "present189"]].to_csv(OUT / "milk_189_by_identifier.csv", index=False); display(blocks)
lad = ladders.set_index("ion"); fam = ["hex_222", "hex_244Na", "hex_186", "hex_168", "hex_144", "hex_138", "hex_126"]
print(f"HexNAc family: XIC correlation with 204.087, median per ion {lad.loc[fam, 'corr_median'].min():.2f}-{lad.loc[fam, 'corr_median'].max():.2f}, minimum over injections {lad.loc[fam, 'corr_min'].min():.2f}; 138.055/204.087 = {lad.loc['hex_138', 'ratio_median']:.2f} "
      f"(IQR {lad.loc['hex_138', 'ratio_q25']:.2f}-{lad.loc['hex_138', 'ratio_q75']:.2f}); PABA 120.044 detected in {100 * lad.loc['paba_120', 'detected_frac']:.0f} % of injections")
print(f"carnitine: [M+Na]+/[M+H]+ = {lad.loc['car_184Na', 'ratio_median']:.4f}, [M+K]+ {lad.loc['car_200K', 'ratio_median']:.3f}, fragment 103.039 {lad.loc['car_103frag', 'ratio_median']:.3f}")

,bin,bin_limits,retained_mz,retained_mz_mean,n_org,n_conv,rt_median,I_median_org,I_median_conv
21,121,"m/z [410,420) x RT [4,6)",412.41,412.414,11,58,5.170,1.788150e+06,1.944708e+06
19,121,"m/z [410,420) x RT [4,6)",410.27,410.266,9,49,5.950,1.829308e+06,1.465466e+06
18,121,"m/z [410,420) x RT [4,6)",410.10,410.099,1,3,4.165,1.940439e+06,1.849331e+06
22,121,"m/z [410,420) x RT [4,6)",414.34,414.336,0,3,5.270,NaN,3.480039e+06
20,121,"m/z [410,420) x RT [4,6)",412.28,412.284,0,2,5.325,NaN,1.391720e+06
26,121,"m/z [410,420) x RT [4,6)",419.31,419.315,2,2,5.280,1.630881e+06,2.130547e+06
27,121,"m/z [410,420) x RT [4,6)",419.32,419.315,9,2,5.320,2.338190e+06,9.487477e+05
23,121,"m/z [410,420) x RT [4,6)",416.14,416.139,1,1,5.620,1.696278e+06,8.569212e+05
24,121,"m/z [410,420) x RT [4,6)",416.37,416.373,1,1,5.080,4.936909e+06,5.721728e+06
25,121,"m/z [410,420) x RT [4,6)",417.18,417.180,8,1,5.390,6.016230e+06,2.045706e+06


,category,batch,n,present
0,b,20220214,7,7
1,b,20220215,13,13
2,b,20220216,24,18
3,c,20220214,6,0
4,c,20220215,54,6
5,c,20220216,62,8


HexNAc family: XIC correlation with 204.087, median per ion 0.91-1.00, minimum over injections 0.36; 138.055/204.087 = 0.36 (IQR 0.29-0.38); PABA 120.044 detected in 0 % of injections
carnitine: [M+Na]+/[M+H]+ = 0.0022, [M+K]+ 0.000, fragment 103.039 0.017


In [17]:
# extracted-ion chromatograms of the HexNAc family and of the Leu/Ile competitor in one conventional injection (panel A of Figure S4)
MILK_INJ = "lai_c_01353_h_r1_22_20220216"
MILK_IONS = {"[M+H]+ 222.097": HEX + H, "[M+Na]+ 244.079": HEX + Na, "oxonium 204.087": 204.08665, "186.076": 186.07608, "138.055 (``PABA'' ion)": 138.05495, "126.055": 126.05495,
             "PABA 120.044": 120.04439, "PABA [M+Na]+ 160.037": PABA + Na, "Leu/Ile [M+H]+ 132.102": LEU + H}
trace_file = OUT / "xic_traces_milk.csv"
hits = list((mn.RAW / "milk").rglob(f"{MILK_INJ}.mzXML")) if mn.raw_available("milk") else []
if hits:
    tr = pd.DataFrame([dict(rt=rt, **{k: peak(mz, it, m) for k, m in MILK_IONS.items()}) for rt, mz, it in read_scans(hits[0], 5.9, 7.6)])
    tr.melt("rt", var_name="ion", value_name="I").assign(panel="A_milk").to_csv(trace_file, index=False)
if trace_file.exists():
    g = pd.read_csv(trace_file); fig, ax = plt.subplots(figsize=(7.6, 3.3))
    styles = {"[M+H]+ 222.097": ("#1f77b4", "-", 1.3), "[M+Na]+ 244.079": ("#17becf", "-", 1.1), "oxonium 204.087": ("#2ca02c", "-", 1.3), "186.076": ("#8c564b", "-", 1.1),
              "138.055 (``PABA'' ion)": ("#d62728", "-", 2.0), "126.055": ("#9467bd", "-", 1.1), "PABA 120.044": ("0.45", ":", 1.6), "PABA [M+Na]+ 160.037": ("0.7", ":", 1.6), "Leu/Ile [M+H]+ 132.102": ("k", "--", 1.3)}
    ref_max = g[g.ion == "oxonium 204.087"].I.max()
    for ion, (c, ls, lw) in styles.items():
        s = g[g.ion == ion]; top = s.I.max() if s.I.max() > 0 else 1
        ax.plot(s.rt, s.I / (top if not ion.startswith("PABA") else ref_max), ls, color=c, lw=lw, label=ion.replace("``", "\u201c").replace("''", "\u201d"))
    ax.set_xlim(5.9, 7.6); ax.set_ylim(0, 1.08); ax.set_xlabel("RT (min)"); ax.set_ylabel("intensity / apex of the ion"); ax.legend(fontsize=6.5, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.02))
    ax.set_title(f"milk, conventional injection {MILK_INJ.split('_h_')[0]} (batch 20220216): HexNAc ion family and the Leu/Ile competitor", fontsize=8.5, loc="left")
    plt.tight_layout(); fig.savefig(FIG / "figS4A_milk_xics.pdf"); fig.savefig(FIG / "figS4A_milk_xics.png", dpi=160); plt.close(fig); print("panel A written")

panel A written


## 9. The transform on one raw file

`minut.transform_file` applied to the first injection (`_r1_`) of three samples, next to the deposited
vector of the same sample (the deposited table holds the first injection of each sample). The milk tables
were computed from these centroided MS$^1$ files. The original run stored the intensity block in single
precision, so the m/z and RT blocks are identical and the intensity block agrees to about 5·10⁻⁸.

In [18]:
if mn.raw_available("milk"):
    done = 0
    for f in [p for p in sorted((mn.RAW / "milk").rglob("*_r1_*.mzXML"))][:3]:
        sid = f.stem.split("_")[2]; row = raw[raw.echantillon.astype(str) == sid]
        if row.empty:
            continue
        v = mn.transform_file(f, GRID, ms_levels=(1,)); dep = np.asarray(row.output_vector.iloc[0], float)
        print(f"{f.stem}: max |difference| intensity {np.abs(v[:NB] - dep[:NB]).max():.1e}, m/z {np.abs(v[NB:2 * NB] - dep[NB:2 * NB]).max():.1e}, RT {np.abs(v[2 * NB:] - dep[2 * NB:]).max():.1e}")
        done += 1
    if not done:
        print("no deposited sample matched the first raw files (identifier format):", raw.echantillon.head(3).tolist())

lai_b_01088_h_r1_31_20220214: max |difference| intensity 1.4e-08, m/z 0.0e+00, RT 0.0e+00


lai_b_01382_h_r1_23_20220214: max |difference| intensity 2.3e-08, m/z 0.0e+00, RT 0.0e+00


lai_c_01229_h_r1_11_20220214: max |difference| intensity 2.9e-08, m/z 0.0e+00, RT 0.0e+00
